# Weather Data Collection for Miami

This notebook pulls historical weather data from multiple sources:
1. NOAA Climate Data Online (CDO) - Primary precipitation data
2. Open-Meteo - Additional weather features
3. MeteoStat - Cross-validation and gap filling

Target: Miami International Airport (25.7617°N, 80.1918°W)

In [1]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import json
import time
import os
from dotenv import load_dotenv
from functools import wraps

# Load environment variables from .env file in project root
env_path = os.path.join(os.path.dirname(os.path.abspath('')), '.env')
load_dotenv(dotenv_path=env_path)

# Miami International Airport coordinates
MIAMI_LAT = 25.7617
MIAMI_LON = -80.1918
MIAMI_STATION_ID = "GHCND:USW00012839"

# Date range
START_DATE = "1948-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")

# Retry configuration
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

def retry_on_failure(max_retries=MAX_RETRIES, delay=RETRY_DELAY):
    """
    Decorator to retry a function on failure.
    
    Args:
        max_retries: Maximum number of retry attempts
        delay: Delay between retries in seconds
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_retries - 1:
                        print(f"Attempt {attempt + 1}/{max_retries} failed: {str(e)}")
                        print(f"Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        print(f"All {max_retries} attempts failed.")
            raise last_exception
        return wrapper
    return decorator

## 1. NOAA Climate Data Online (CDO)

**Setup Required:**
1. Get API token from: https://www.ncdc.noaa.gov/cdo-web/token
2. Set environment variable: `export NOAA_TOKEN=your_token_here`

**Rate Limits:** 5 requests/second, 10,000 requests/day

In [2]:
# NOAA CDO API Configuration
NOAA_TOKEN = os.getenv('NOAA_TOKEN')
if not NOAA_TOKEN:
    print("WARNING: NOAA_TOKEN not set. Get token from https://www.ncdc.noaa.gov/cdo-web/token")
    print("Then set it: export NOAA_TOKEN=your_token_here")

NOAA_BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"
NOAA_HEADERS = {"token": NOAA_TOKEN}

def fetch_noaa_data(station_id, start_date, end_date, dataset_id="GHCND", limit=1000):
    """
    Fetch data from NOAA CDO API with automatic retry on individual request failures.
    
    Args:
        station_id: Station ID (e.g., 'GHCND:USW00012839')
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        dataset_id: Dataset ID (default: GHCND)
        limit: Results per request (max 1000)
    
    Returns:
        DataFrame with weather data
    """
    all_data = []
    offset = 1
    
    while True:
        params = {
            "datasetid": dataset_id,
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "limit": limit,
            "offset": offset,
            "units": "metric"
        }
        
        # Retry individual request with exponential backoff
        success = False
        for attempt in range(MAX_RETRIES):
            try:
                response = requests.get(
                    f"{NOAA_BASE_URL}/data", 
                    headers=NOAA_HEADERS, 
                    params=params, 
                    timeout=30
                )
                
                if response.status_code != 200:
                    error_msg = f"NOAA API error: {response.status_code} - {response.text}"
                    print(error_msg)
                    raise Exception(error_msg)
                
                data = response.json()
                success = True
                break
                
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                    print(f"Request failed (attempt {attempt + 1}/{MAX_RETRIES}): {str(e)}")
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"All {MAX_RETRIES} attempts failed for offset {offset}")
                    raise
        
        if not success:
            break
        
        if "results" not in data:
            break
        
        all_data.extend(data["results"])
        print(f"Fetched {len(all_data)} records... (offset: {offset})")
        
        # Check if there are more results
        if len(data["results"]) < limit:
            break
        
        offset += limit
        time.sleep(0.5)  # Rate limiting: increased to 2 requests/second to be safe
    
    df = pd.DataFrame(all_data)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'])
        # Pivot to get datatype as columns
        df_pivot = df.pivot_table(index='date', columns='datatype', values='value', aggfunc='first')
        df_pivot.reset_index(inplace=True)
    else:
        df_pivot = pd.DataFrame()
    
    return df_pivot

In [3]:
# Fetch NOAA data in chunks (1 year at a time to avoid hitting limits)
def fetch_noaa_by_year(station_id, start_year=1948, end_year=None):
    """
    Fetch NOAA data year by year to avoid timeout/limit issues.
    """
    if end_year is None:
        end_year = datetime.now().year
    
    all_dfs = []
    
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"
        
        print(f"\nFetching {year}...")
        df_year = fetch_noaa_data(station_id, start_date, end_date)
        
        if not df_year.empty:
            all_dfs.append(df_year)
        
        time.sleep(1.0)  # Increased delay between years
    
    if all_dfs:
        df_combined = pd.concat(all_dfs, ignore_index=True)
        return df_combined
    else:
        return pd.DataFrame()

# Create data directories if they don't exist (relative to project root)
os.makedirs('../data/noaa', exist_ok=True)
os.makedirs('../data/open-meteo', exist_ok=True)
os.makedirs('../data/meteostat', exist_ok=True)

# Fetch from START_DATE (1948) to now
if NOAA_TOKEN:
    print("Fetching NOAA data for Miami International Airport...")
    start_year = int(START_DATE.split('-')[0])
    end_year = datetime.now().year
    print(f"Date range: {start_year} to {end_year}")
    df_noaa = fetch_noaa_by_year(MIAMI_STATION_ID, start_year=start_year, end_year=end_year)
    
    # Save to CSV
    df_noaa.to_csv('../data/noaa/miami_ghcnd.csv', index=False)
    print(f"\nNOAA data saved: {len(df_noaa)} records")
    print(f"Columns: {df_noaa.columns.tolist()}")
    display(df_noaa.head())
else:
    print("Skipping NOAA fetch - token not configured")

Fetching NOAA data for Miami International Airport...
Date range: 1948 to 2026

Fetching 1948...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 2171 records... (offset: 2001)

Fetching 1949...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 2522 records... (offset: 2001)

Fetching 1950...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Fetched 3228 records... (offset: 3001)

Fetching 1951...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Fetched 3247 records... (offset: 3001)

Fetching 1952...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Fetched 3248 records... (offset: 3001)

Fetching 1953...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Fetched 3277

datatype,date,PRCP,SNOW,SNWD,TMAX,TMIN,TOBS,WT01,WT03,WT08,...,WT04,WT21,ADPT,ASLP,ASTP,AWBT,RHAV,RHMN,RHMX,WT18
0,1948-01-01,0.0,0.0,0.0,27.2,20.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1948-01-02,3.6,0.0,0.0,27.2,20.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1948-01-03,0.0,0.0,0.0,22.2,15.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1948-01-04,0.0,0.0,0.0,22.2,13.9,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1948-01-05,0.0,0.0,0.0,18.9,8.9,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Open-Meteo Historical Weather API

**No API key required!**
- Data from 1940 to present (5-day delay)
- Hourly resolution
- Very fast (decades of data in seconds)

In [4]:
def fetch_open_meteo_data(lat, lon, start_date, end_date, hourly_vars=None, daily_vars=None):
    """
    Fetch historical weather data from Open-Meteo with automatic retry on failure.
    
    Args:
        lat: Latitude
        lon: Longitude
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        hourly_vars: List of hourly variables to fetch
        daily_vars: List of daily variables to fetch
    
    Returns:
        Tuple of (hourly_df, daily_df)
    """
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "America/New_York"
    }
    
    if hourly_vars:
        params["hourly"] = ",".join(hourly_vars)
    
    if daily_vars:
        params["daily"] = ",".join(daily_vars)
    
    print(f"Fetching Open-Meteo data from {start_date} to {end_date}...")
    
    # Retry with exponential backoff
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(base_url, params=params, timeout=30)
            
            if response.status_code != 200:
                error_msg = f"Open-Meteo API error: {response.status_code} - {response.text}"
                print(error_msg)
                raise Exception(error_msg)
            
            data = response.json()
            break
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                print(f"Request failed (attempt {attempt + 1}/{MAX_RETRIES}): {str(e)}")
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"All {MAX_RETRIES} attempts failed.")
                raise
    
    hourly_df = None
    daily_df = None
    
    if "hourly" in data:
        hourly_df = pd.DataFrame(data["hourly"])
        hourly_df['time'] = pd.to_datetime(hourly_df['time'])
    
    if "daily" in data:
        daily_df = pd.DataFrame(data["daily"])
        daily_df['time'] = pd.to_datetime(daily_df['time'])
    
    return hourly_df, daily_df

In [5]:
# Define variables to fetch
hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm"
]

daily_variables = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "sunshine_duration"
]

# Fetch data from START_DATE to END_DATE
print(f"Fetching Open-Meteo data from {START_DATE} to {END_DATE}")
hourly_df, daily_df = fetch_open_meteo_data(
    MIAMI_LAT, 
    MIAMI_LON, 
    START_DATE,
    END_DATE,
    hourly_vars=hourly_variables,
    daily_vars=daily_variables
)

if hourly_df is not None:
    hourly_df.to_csv('../data/open-meteo/miami_hourly.csv', index=False)
    print(f"\nOpen-Meteo hourly data saved: {len(hourly_df)} records")
    display(hourly_df.head())

if daily_df is not None:
    daily_df.to_csv('../data/open-meteo/miami_daily.csv', index=False)
    print(f"\nOpen-Meteo daily data saved: {len(daily_df)} records")
    display(daily_df.head())

Fetching Open-Meteo data from 1948-01-01 to 2026-01-22
Fetching Open-Meteo data from 1948-01-01 to 2026-01-22...
Open-Meteo API error: 500 - {"reason":"Something went wrong.","error":true}
Request failed (attempt 1/3): Open-Meteo API error: 500 - {"reason":"Something went wrong.","error":true}
Retrying in 2 seconds...

Open-Meteo hourly data saved: 684288 records


,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm
0,1948-01-01 00:00:00,22.7,74,17.9,0.0,0.0,1016.6,1016.0,54,18.2,141,26.6,22.4,0.067
1,1948-01-01 01:00:00,22.7,75,18.1,0.0,0.0,1016.1,1015.5,58,19.0,147,27.0,22.4,0.067
2,1948-01-01 02:00:00,22.7,77,18.4,0.0,0.0,1016.0,1015.4,54,18.7,150,26.6,22.4,0.068
3,1948-01-01 03:00:00,22.6,79,18.7,0.0,0.0,1015.5,1014.9,51,17.7,153,25.2,22.4,0.068
4,1948-01-01 04:00:00,22.4,81,19.0,0.0,0.0,1015.2,1014.6,54,15.1,155,21.2,22.4,0.068



Open-Meteo daily data saved: 28512 records


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,sunshine_duration
0,1948-01-01,25.7,22.4,0.6,0.6,5.0,27.0,40.3,34472.54
1,1948-01-02,25.0,20.2,1.8,1.8,8.0,23.3,43.9,31360.01
2,1948-01-03,21.6,16.6,0.0,0.0,0.0,18.1,27.4,34627.13
3,1948-01-04,21.6,16.2,0.0,0.0,0.0,21.7,34.2,34043.72
4,1948-01-05,18.6,12.9,0.0,0.0,0.0,22.8,38.9,35078.13


## 3. MeteoStat

**Installation:** `pip install meteostat`
- Uses NOAA and other sources
- Simple Python interface
- Returns Pandas DataFrames

In [6]:
# Install meteostat if not already installed
try:
    from meteostat import Point, Daily, Hourly
except ImportError:
    print("Installing meteostat...")
    !pip install meteostat
    from meteostat import Point, Daily, Hourly

Installing meteostat...

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


ImportError: cannot import name 'Daily' from 'meteostat' (/Users/justinbonner/Spring2026/CSC542/GroupProject/.venv/lib/python3.12/site-packages/meteostat/__init__.py)

In [7]:
# Create location point
miami_point = Point(MIAMI_LAT, MIAMI_LON)

# Fetch daily data with retry logic
@retry_on_failure(max_retries=3, delay=2)
def fetch_meteostat_daily(point, start, end):
    """Fetch MeteoStat daily data with retry on failure."""
    print("Fetching MeteoStat daily data...")
    daily_data = Daily(point, start=start, end=end)
    df = daily_data.fetch()
    if df.empty:
        raise Exception("MeteoStat returned empty dataframe")
    return df

# Parse START_DATE and END_DATE
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt = datetime.strptime(END_DATE, "%Y-%m-%d")

print(f"Fetching MeteoStat daily data from {START_DATE} to {END_DATE}")
df_meteostat_daily = fetch_meteostat_daily(
    miami_point,
    start=start_dt,
    end=end_dt
)

# Save to CSV
df_meteostat_daily.to_csv('../data/meteostat/miami_daily.csv')
print(f"\nMeteoStat daily data saved: {len(df_meteostat_daily)} records")
print(f"Columns: {df_meteostat_daily.columns.tolist()}")
display(df_meteostat_daily.head())

Fetching MeteoStat daily data from 1948-01-01 to 2026-01-22
Fetching MeteoStat daily data...
Attempt 1/3 failed: name 'Daily' is not defined
Retrying in 2 seconds...
Fetching MeteoStat daily data...
Attempt 2/3 failed: name 'Daily' is not defined
Retrying in 2 seconds...
Fetching MeteoStat daily data...
All 3 attempts failed.


NameError: name 'Daily' is not defined

In [11]:
# Fetch hourly data (optional - can be large for long periods)
# NOTE: Hourly data for 75+ years will be VERY large (650,000+ records)
# Consider reducing the date range if needed
@retry_on_failure(max_retries=3, delay=2)
def fetch_meteostat_hourly(point, start, end):
    """Fetch MeteoStat hourly data with retry on failure."""
    print("Fetching MeteoStat hourly data...")
    hourly_data = Hourly(point, start=start, end=end)
    df = hourly_data.fetch()
    if df.empty:
        raise Exception("MeteoStat returned empty dataframe")
    return df

# For hourly data, use a more reasonable range (last 2 years) to avoid huge files
# Uncomment and modify dates as needed for your use case
hourly_start = datetime(2023, 1, 1)
hourly_end = datetime.now()

print(f"WARNING: Fetching hourly data for full range ({START_DATE} to {END_DATE}) would be very large.")
print(f"Using reduced range: {hourly_start} to {hourly_end}")

df_meteostat_hourly = fetch_meteostat_hourly(
    miami_point,
    start=hourly_start,
    end=hourly_end
)

# Save to CSV
df_meteostat_hourly.to_csv('../data/meteostat/miami_hourly.csv')
print(f"\nMeteoStat hourly data saved: {len(df_meteostat_hourly)} records")
print(f"Columns: {df_meteostat_hourly.columns.tolist()}")
display(df_meteostat_hourly.head())

Using reduced range: 2023-01-01 00:00:00 to 2026-01-23 09:35:28.440735
Fetching MeteoStat hourly data...
Attempt 1/3 failed: name 'Hourly' is not defined
Retrying in 2 seconds...
Fetching MeteoStat hourly data...
Attempt 2/3 failed: name 'Hourly' is not defined
Retrying in 2 seconds...
Fetching MeteoStat hourly data...
All 3 attempts failed.


NameError: name 'Hourly' is not defined

## 4. Data Consolidation

Merge data from all sources into a single dataset.

In [8]:
# Load saved data
try:
    df_noaa = pd.read_csv('../data/noaa/miami_ghcnd.csv', parse_dates=['date'])
    print(f"NOAA data loaded: {len(df_noaa)} records")
except FileNotFoundError:
    df_noaa = pd.DataFrame()
    print("NOAA data not found")

try:
    df_open_meteo = pd.read_csv('../data/open-meteo/miami_daily.csv', parse_dates=['time'])
    df_open_meteo.rename(columns={'time': 'date'}, inplace=True)
    print(f"Open-Meteo data loaded: {len(df_open_meteo)} records")
except FileNotFoundError:
    df_open_meteo = pd.DataFrame()
    print("Open-Meteo data not found")

try:
    df_meteostat = pd.read_csv('../data/meteostat/miami_daily.csv', parse_dates=['time'])
    df_meteostat.rename(columns={'time': 'date'}, inplace=True)
    print(f"MeteoStat data loaded: {len(df_meteostat)} records")
except FileNotFoundError:
    df_meteostat = pd.DataFrame()
    print("MeteoStat data not found")

NOAA data loaded: 28508 records
Open-Meteo data loaded: 28512 records
MeteoStat data not found


In [9]:
# Merge datasets on date
df_consolidated = pd.DataFrame()

# Start with NOAA as base (precipitation target)
if not df_noaa.empty:
    df_consolidated = df_noaa.copy()
    df_consolidated = df_consolidated.add_prefix('noaa_')
    df_consolidated.rename(columns={'noaa_date': 'date'}, inplace=True)

# Merge Open-Meteo
if not df_open_meteo.empty:
    df_open_meteo_prefixed = df_open_meteo.add_prefix('openmeteo_')
    df_open_meteo_prefixed.rename(columns={'openmeteo_date': 'date'}, inplace=True)
    
    if df_consolidated.empty:
        df_consolidated = df_open_meteo_prefixed
    else:
        df_consolidated = pd.merge(df_consolidated, df_open_meteo_prefixed, on='date', how='outer')

# Merge MeteoStat
if not df_meteostat.empty:
    df_meteostat_prefixed = df_meteostat.add_prefix('meteostat_')
    df_meteostat_prefixed.rename(columns={'meteostat_date': 'date'}, inplace=True)
    
    if df_consolidated.empty:
        df_consolidated = df_meteostat_prefixed
    else:
        df_consolidated = pd.merge(df_consolidated, df_meteostat_prefixed, on='date', how='outer')

# Sort by date
if not df_consolidated.empty:
    df_consolidated.sort_values('date', inplace=True)
    df_consolidated.reset_index(drop=True, inplace=True)
    
    # Save consolidated dataset
    df_consolidated.to_csv('../data/miami_consolidated.csv', index=False)
    print(f"\nConsolidated dataset saved: {len(df_consolidated)} records")
    print(f"Date range: {df_consolidated['date'].min()} to {df_consolidated['date'].max()}")
    print(f"\nColumns ({len(df_consolidated.columns)}): {df_consolidated.columns.tolist()}")
    display(df_consolidated.head())
    
    # Data quality summary
    print("\n=== Data Completeness ===")
    print(df_consolidated.isnull().sum())
else:
    print("No data to consolidate")


Consolidated dataset saved: 28512 records
Date range: 1948-01-01 00:00:00 to 2026-01-22 00:00:00

Columns (55): ['date', 'noaa_PRCP', 'noaa_SNOW', 'noaa_SNWD', 'noaa_TMAX', 'noaa_TMIN', 'noaa_TOBS', 'noaa_WT01', 'noaa_WT03', 'noaa_WT08', 'noaa_WT16', 'noaa_PGTM', 'noaa_WDFG', 'noaa_WSFG', 'noaa_WT05', 'noaa_WT07', 'noaa_ACMH', 'noaa_ACSH', 'noaa_WDF1', 'noaa_WSF1', 'noaa_WT02', 'noaa_PSUN', 'noaa_TSUN', 'noaa_WT09', 'noaa_WESD', 'noaa_WT06', 'noaa_AWND', 'noaa_FMTM', 'noaa_WT14', 'noaa_WT10', 'noaa_WDF2', 'noaa_WDF5', 'noaa_WSF2', 'noaa_WSF5', 'noaa_WT11', 'noaa_WT13', 'noaa_TAVG', 'noaa_WT04', 'noaa_WT21', 'noaa_ADPT', 'noaa_ASLP', 'noaa_ASTP', 'noaa_AWBT', 'noaa_RHAV', 'noaa_RHMN', 'noaa_RHMX', 'noaa_WT18', 'openmeteo_temperature_2m_max', 'openmeteo_temperature_2m_min', 'openmeteo_precipitation_sum', 'openmeteo_rain_sum', 'openmeteo_precipitation_hours', 'openmeteo_wind_speed_10m_max', 'openmeteo_wind_gusts_10m_max', 'openmeteo_sunshine_duration']


,date,noaa_PRCP,noaa_SNOW,noaa_SNWD,noaa_TMAX,noaa_TMIN,noaa_TOBS,noaa_WT01,noaa_WT03,noaa_WT08,...,noaa_RHMX,noaa_WT18,openmeteo_temperature_2m_max,openmeteo_temperature_2m_min,openmeteo_precipitation_sum,openmeteo_rain_sum,openmeteo_precipitation_hours,openmeteo_wind_speed_10m_max,openmeteo_wind_gusts_10m_max,openmeteo_sunshine_duration
0,1948-01-01,0.0,0.0,0.0,27.2,20.6,NaN,NaN,NaN,NaN,...,NaN,NaN,25.7,22.4,0.6,0.6,5.0,27.0,40.3,34472.54
1,1948-01-02,3.6,0.0,0.0,27.2,20.6,NaN,NaN,NaN,NaN,...,NaN,NaN,25.0,20.2,1.8,1.8,8.0,23.3,43.9,31360.01
2,1948-01-03,0.0,0.0,0.0,22.2,15.6,NaN,NaN,NaN,NaN,...,NaN,NaN,21.6,16.6,0.0,0.0,0.0,18.1,27.4,34627.13
3,1948-01-04,0.0,0.0,0.0,22.2,13.9,NaN,NaN,NaN,NaN,...,NaN,NaN,21.6,16.2,0.0,0.0,0.0,21.7,34.2,34043.72
4,1948-01-05,0.0,0.0,0.0,18.9,8.9,NaN,NaN,NaN,NaN,...,NaN,NaN,18.6,12.9,0.0,0.0,0.0,22.8,38.9,35078.13



=== Data Completeness ===
date                                 0
noaa_PRCP                            4
noaa_SNOW                         6192
noaa_SNWD                         6363
noaa_TMAX                            4
noaa_TMIN                            4
noaa_TOBS                        28481
noaa_WT01                        23906
noaa_WT03                        22685
noaa_WT08                        24941
noaa_WT16                        16394
noaa_PGTM                         6498
noaa_WDFG                        11434
noaa_WSFG                        11425
noaa_WT05                        28133
noaa_WT07                        28486
noaa_ACMH                        17008
noaa_ACSH                        17008
noaa_WDF1                        17008
noaa_WSF1                        17008
noaa_WT02                        28226
noaa_PSUN                        24539
noaa_TSUN                        18688
noaa_WT09                        28502
noaa_WESD                        2266

## 5. Quick Data Exploration

In [10]:
if not df_consolidated.empty:
    # Basic statistics
    print("=== Dataset Statistics ===")
    display(df_consolidated.describe())
    
    # Check for precipitation columns
    precip_cols = [col for col in df_consolidated.columns if 'prcp' in col.lower() or 'precipitation' in col.lower()]
    print(f"\n=== Precipitation Columns ===")
    print(precip_cols)
    
    if precip_cols:
        display(df_consolidated[['date'] + precip_cols].head(20))

=== Dataset Statistics ===


,date,noaa_PRCP,noaa_SNOW,noaa_SNWD,noaa_TMAX,noaa_TMIN,noaa_TOBS,noaa_WT01,noaa_WT03,noaa_WT08,...,noaa_RHMX,noaa_WT18,openmeteo_temperature_2m_max,openmeteo_temperature_2m_min,openmeteo_precipitation_sum,openmeteo_rain_sum,openmeteo_precipitation_hours,openmeteo_wind_speed_10m_max,openmeteo_wind_gusts_10m_max,openmeteo_sunshine_duration
count,28512,28508.000000,22320.0,22149.000000,28508.000000,28508.000000,31.000000,4606.0,5827.0,3571.0,...,6878.000000,1.0,28512.000000,28512.000000,28512.000000,28512.000000,28512.000000,28512.000000,28512.000000,28512.000000
mean,1987-01-11 12:00:00.000000064,4.310495,0.0,0.010339,28.730248,20.950754,28.690323,1.0,1.0,1.0,...,86.071823,1.0,26.846033,22.146679,2.945441,2.945441,5.744879,21.277132,36.321233,36746.961794
min,1948-01-01 00:00:00,0.000000,0.0,0.000000,7.200000,-1.100000,26.100000,1.0,1.0,1.0,...,50.000000,1.0,8.000000,2.200000,0.000000,0.000000,0.000000,6.200000,13.000000,0.000000
25%,1967-07-07 18:00:00,0.000000,0.0,0.000000,26.700000,18.900000,28.300000,1.0,1.0,1.0,...,81.000000,1.0,25.100000,20.500000,0.000000,0.000000,0.000000,16.500000,29.200000,34443.260000
50%,1987-01-11 12:00:00,0.000000,0.0,0.000000,29.400000,22.200000,28.900000,1.0,1.0,1.0,...,87.000000,1.0,27.300000,23.000000,0.700000,0.700000,4.000000,20.400000,34.900000,38068.645000
75%,2006-07-18 06:00:00,1.800000,0.0,0.000000,31.700000,24.400000,28.900000,1.0,1.0,1.0,...,93.000000,1.0,29.200000,25.000000,3.100000,3.100000,10.000000,25.200000,42.100000,42380.940000
max,2026-01-22 00:00:00,377.200000,0.0,229.000000,36.700000,28.900000,31.100000,1.0,1.0,1.0,...,100.000000,1.0,36.200000,28.500000,213.300000,213.300000,24.000000,124.100000,195.100000,46313.380000
std,NaN,12.891259,0.0,1.538716,3.618419,4.644720,1.121711,0.0,0.0,0.0,...,8.308580,NaN,3.074738,3.803754,6.494816,6.494816,6.219321,6.548150,9.927017,8382.213155



=== Precipitation Columns ===
['noaa_PRCP', 'openmeteo_precipitation_sum', 'openmeteo_precipitation_hours']


,date,noaa_PRCP,openmeteo_precipitation_sum,openmeteo_precipitation_hours
0,1948-01-01,0.0,0.6,5.0
1,1948-01-02,3.6,1.8,8.0
2,1948-01-03,0.0,0.0,0.0
3,1948-01-04,0.0,0.0,0.0
4,1948-01-05,0.0,0.0,0.0
5,1948-01-06,0.0,0.0,0.0
6,1948-01-07,0.0,0.0,0.0
7,1948-01-08,0.0,0.0,0.0
8,1948-01-09,0.0,0.0,0.0
9,1948-01-10,0.0,0.0,0.0


## Next Steps

1. **Extend date range**: Modify `start_year` in NOAA fetch to go back to 1948
2. **Feature engineering**: Create lagged features, rolling averages, seasonal indicators
3. **Handle missing data**: Interpolation, forward fill, or deletion strategies
4. **Target variable**: Define precipitation threshold for classification (e.g., >0mm, >1mm, >10mm)
5. **Add Kalshi data**: When available, add market predictions as features